# Warehouse Grid Model & Loader — Explanation + Demo

This notebook explains the **warehouse grid model** you implemented (cells, floors, building, connectors, occupancy), and provides small runnable demos (rendering, occupancy, region queries, and shortest-path using Dijkstra).

_Generated on 2026-02-13._

## 0) What this codebase is for

You built a **grid-based representation of a warehouse**:

- Every floor is a 2‑D grid (1 cell = **1m × 1m**).
- Each cell has a **Kind** (VOID/OBSTACLE/CORRIDOR/STORAGE/CONNECTOR) plus runtime state (**occupied**, **blocked**).
- Multiple floors are grouped into a **WarehouseBuilding**.
- **Connectors** (monte-charge / elevator cells) create **vertical edges** between floors.
- Slot/zone labels like `C7`, `E14`, ... are supported via a `slot_code` stamped on storage cells from region maps.

This is the “world model” needed for later services like:
- **Storage optimization** (choose a free slot that’s “best” by distance + constraints).
- **Picking optimization** (compute shortest routes for pick lists).


In [1]:
from pathlib import Path
import os, json, yaml

def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "config").exists() and (parent / "warehouse").exists():
            return parent
    raise FileNotFoundError("Could not find project root containing both 'config/' and 'warehouse/'")

PROJECT_ROOT = find_project_root()
print("PROJECT_ROOT =", PROJECT_ROOT)

floors_cfg_path     = PROJECT_ROOT / "config" / "floors.updated.yaml"
connectors_cfg_path = PROJECT_ROOT / "config" / "connectors.yaml"
regions12_path      = PROJECT_ROOT / "config" / "slot_regions_floor12.updated.json"
regions34_path      = PROJECT_ROOT / "config" / "slot_regions_floor34.updated.json"

# sanity checks
for p in [floors_cfg_path, connectors_cfg_path, regions12_path, regions34_path]:
    print(p, "->", p.exists())


PROJECT_ROOT = /home/hai_bou.19/bms/ai/haithem
/home/hai_bou.19/bms/ai/haithem/config/floors.updated.yaml -> True
/home/hai_bou.19/bms/ai/haithem/config/connectors.yaml -> True
/home/hai_bou.19/bms/ai/haithem/config/slot_regions_floor12.updated.json -> True
/home/hai_bou.19/bms/ai/haithem/config/slot_regions_floor34.updated.json -> True


## 2) Core types: `Kind` + `Cell`

### `Kind`
A `Kind` classifies what a cell represents:
- `VOID`: outside the building envelope
- `OBSTACLE`: pillars/walls/rack frame (never walkable)
- `CORRIDOR`: aisle (walkable)
- `STORAGE`: pallet footprint (walkability depends on occupancy policy)
- `CONNECTOR`: vertical movement node (monte-charge / elevator)

### `Cell`
A `Cell` has:
- **Static/layout** fields: `kind`, `connector_id`, `slot_code`, `capacity`, `base_cost`
- **Dynamic/runtime** fields: `occupied`, `blocked`

So the same map can be reused, while occupancy changes over time.


In [2]:
# Quick peek at the dataclasses
import inspect
print(inspect.getsource(Kind)[:800], "...")
print("\n---\n")
print(inspect.getsource(Cell)[:800], "...")

NameError: name 'Kind' is not defined

## 3) Walkability policy + movement cost

You implemented **Policy C** for pathfinding:

- VOID / OBSTACLE → never walkable
- blocked cells → never walkable
- CORRIDOR + CONNECTOR → walkable
- STORAGE → walkable **only if empty** (`occupied == False`)

Movement cost:
- Entering an empty STORAGE cell costs `base_cost + storage_penalty`
- Entering other walkable cells costs `base_cost`

This encourages paths to prefer corridors but still allows cutting through empty storage when useful.


In [3]:
# Demonstrate walkable() + move_cost()
samples = [
    Cell(kind=Kind.VOID),
    Cell(kind=Kind.OBSTACLE),
    Cell(kind=Kind.CORRIDOR),
    Cell(kind=Kind.STORAGE, occupied=False),
    Cell(kind=Kind.STORAGE, occupied=True),
    Cell(kind=Kind.CONNECTOR),
]

for i, cell in enumerate(samples, 1):
    print(i, cell.kind, "occupied=", cell.occupied, "blocked=", cell.blocked,
          "walkable=", walkable(cell),
          "cost_to_enter=", move_cost(cell, storage_penalty=2.0))

1 Kind.VOID occupied= False blocked= False walkable= False cost_to_enter= 1.0
2 Kind.OBSTACLE occupied= False blocked= False walkable= False cost_to_enter= 1.0
3 Kind.CORRIDOR occupied= False blocked= False walkable= True cost_to_enter= 1.0
4 Kind.STORAGE occupied= False blocked= False walkable= True cost_to_enter= 3.0
5 Kind.STORAGE occupied= True blocked= False walkable= False cost_to_enter= 3.0
6 Kind.CONNECTOR occupied= False blocked= False walkable= True cost_to_enter= 1.0


## 4) Grid containers

### `WarehouseFloorGrid`
- Holds a 2‑D matrix `cells[r][c]` for one floor
- Provides helpers: `in_bounds()`, indexing `grid[r,c]`, and `summary()`

### `WarehouseBuilding`
- Holds `floors: dict[int, WarehouseFloorGrid]`
- Holds `connector_targets: dict[connector_id -> list[(floor,r,c)]]`
- Provides `neighbors(floor,r,c)` that yields:
  - 4‑connected horizontal moves (N/S/E/W)
  - vertical moves via connectors (jump to same connector on other floors)


In [4]:
# Show the neighbors() signature
import inspect
print(inspect.getsource(WarehouseBuilding.neighbors)[:1200], "...")

    def neighbors(
        self,
        floor: int,
        r: int,
        c: int,
    ) -> Iterator[tuple[int, int, int, float]]:
        """
        Yield reachable neighbours of cell ``(floor, r, c)``.

        Yields
        ------
        (next_floor, next_row, next_col, step_cost)

        Movement rules:
            1. **Horizontal** — 4-connected (N/S/E/W) on the same floor.
               Target must be walkable.
            2. **Vertical** — only if the current cell is a CONNECTOR.
               Jumps to every other floor where the same connector_id
               exists.  Cost = |Δfloor| × vertical_cost_per_floor.
        """
        grid = self.floors.get(floor)
        if grid is None:
            return

        current = grid[r, c]

        # ── 1) horizontal 4-neighbourhood ─────────────────
        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            nr, nc = r + dr, c + dc
            if not grid.in_bounds(nr, nc):
                continue
            targ

## 5) Loader pipeline: ASCII → grids → regions → connectors → building

Your loader is a **factory** that assembles the building from config + files.

### A) `load_plan_from_ascii(ascii_str, floor, ...)`
Reads a multi-line ASCII string where each character maps to a cell Kind:
- `" "` -> VOID
- `"."` -> CORRIDOR
- `"S"` -> STORAGE
- `"#"` -> OBSTACLE
- `"E"`, `"M"` -> CONNECTOR (config can override connector IDs)

It pads shorter lines, so the grid is rectangular.

### B) `apply_slot_regions(grid, slot_regions)`
Takes a dict like:
```json
{ "C7": [[2,3],[2,4]], "E14": [[5,8],[5,9]] }
```
and stamps `slot_code="C7"` etc into those cells.

### C) `apply_connectors(building, connectors_config)`
Reads a list of connector entries (monte-charge / elevator) and:
- forces those cells to `Kind.CONNECTOR`
- sets `cell.connector_id`
- fills `building.connector_targets[cid] = [(floor,r,c), ...]`
- can override the building-wide `vertical_cost_per_floor`

### D) `build_building(config)`
One-call factory assembling floors 1..4:
- Floors (1,2) share `plan12`
- Floors (3,4) share `plan34`
- then slot regions, then connectors


## 6) Export / debugging helpers

### `render_floor_ascii(grid, show_occupied=False)`
Renders the grid back to ASCII for debugging.
If `show_occupied=True`:
- occupied storage becomes `X`
- blocked cells become `!`

### `print_building_summary(building)`
Prints counts of cell kinds and connector positions.


## 7) Occupancy helpers

You added thin helpers so that later you can plug a database (e.g. Supabase stock table) without changing pathfinding logic:

- `set_occupied(building, floor, r, c, occupied=True)`  
  Only valid for STORAGE cells (raises error otherwise)

- `set_blocked(building, floor, r, c, blocked=True)`  
  Temporarily disable a cell (maintenance)

- `get_free_cells_in_region(grid, slot_code)`  
  Returns all STORAGE cells that belong to `slot_code` and are **not occupied** and **not blocked**


## 8) Demo with a small toy warehouse (2 floors)

We’ll build a tiny example that contains:
- corridors (`.`)
- storage (`S`)
- obstacles (`#`)
- connectors (`M`) to move vertically between floors

> Your real project uses 4 floors and reads plans + connectors from files. This demo is just to make behavior obvious.


In [5]:
# Build a small toy building with 2 floors using build_building()

toy_plan = """
#########
#..S..M.#
#..S....#
#..S..S.#
#########
"""  # M is a connector symbol

config = {
    "plan12": toy_plan,      # we will reuse same plan for simplicity
    "plan34": toy_plan,      # unused in this toy but required by API
    "storage_penalty": 2.0,
    "vertical_cost_per_floor": 7.0,
    "connector_symbols": {"M": "mc1"},  # map 'M' -> connector id mc1
    "connectors": [
        {
            "id": "mc1",
            "positions": {"1": [1, 6], "2": [1, 6]},  # (row=1,col=6) on floor 1 & 2
            "vertical_cost_per_floor": 7.0,
        }
    ],
    "slot_regions_12": None,
    "slot_regions_34": None,
}

building = build_building(config)

print_building_summary(building)

print("\n--- Floor 1 (initial) ---")
print(render_floor_ascii(building.floors[1]))
print("\n--- Floor 2 (initial) ---")
print(render_floor_ascii(building.floors[2]))


═══ Floor 1 (5×9 = 45 cells) ═══
  OBSTACLE    :    24
  CORRIDOR    :    16
  STORAGE     :     4
  CONNECTOR   :     1

═══ Floor 2 (5×9 = 45 cells) ═══
  OBSTACLE    :    24
  CORRIDOR    :    16
  STORAGE     :     4
  CONNECTOR   :     1

═══ Floor 3 (5×9 = 45 cells) ═══
  OBSTACLE    :    24
  CORRIDOR    :    16
  STORAGE     :     4
  CONNECTOR   :     1

═══ Floor 4 (5×9 = 45 cells) ═══
  OBSTACLE    :    24
  CORRIDOR    :    16
  STORAGE     :     4
  CONNECTOR   :     1

═══ Connectors ═══
  mc1: F1@(1,6), F2@(1,6)

--- Floor 1 (initial) ---
#########
#..S..E.#
#..S....#
#..S..S.#
#########

--- Floor 2 (initial) ---
#########
#..S..E.#
#..S....#
#..S..S.#
#########


In [6]:
# Mark some storage as occupied and a corridor cell as blocked

# Occupy a storage at (floor=1,row=1,col=3) (that's the 'S' in row 1 col 3)
set_occupied(building, 1, 1, 3, True)

# Block a corridor cell at (floor=1,row=1,col=2) (that's a '.')
set_blocked(building, 1, 1, 2, True)

print("--- Floor 1 (show_occupied=True) ---")
print(render_floor_ascii(building.floors[1], show_occupied=True))

--- Floor 1 (show_occupied=True) ---
#########
#.!X..E.#
#..S....#
#..S..S.#
#########


In [7]:
# Region stamping demo (slot_code)
# We'll create a fake region called 'C7' and assign it to a couple of storage cells.

slot_regions = {
    "C7": [[1, 3], [2, 3], [3, 3]],  # the vertical storage column
    "E14": [[3, 6]],                 # another storage cell
}
apply_slot_regions(building.floors[1], slot_regions)

# Query free cells in region C7 on floor 1 (should exclude occupied/blocked)
free_c7 = get_free_cells_in_region(building.floors[1], "C7")
free_e14 = get_free_cells_in_region(building.floors[1], "E14")
print("Free cells in C7:", free_c7)
print("Free cells in E14:", free_e14)

Free cells in C7: [(2, 3), (3, 3)]
Free cells in E14: [(3, 6)]


## 9) Shortest path demo (Dijkstra) using `building.neighbors()`

Your codebase provides the **graph edges** via `neighbors(...)`.
To compute paths you can plug in A* or Dijkstra.

Below is a minimal Dijkstra that finds the cheapest path cost
between two nodes `(floor,row,col)`.


In [8]:
import heapq
from math import inf

def dijkstra(building, start, goal):
    pq = [(0.0, start)]
    dist = {start: 0.0}
    prev = {start: None}

    while pq:
        d, node = heapq.heappop(pq)
        if d != dist.get(node, inf):
            continue
        if node == goal:
            break

        f, r, c = node
        for nf, nr, nc, step_cost in building.neighbors(f, r, c):
            nxt = (nf, nr, nc)
            nd = d + step_cost
            if nd < dist.get(nxt, inf):
                dist[nxt] = nd
                prev[nxt] = node
                heapq.heappush(pq, (nd, nxt))

    if goal not in dist:
        return inf, []

    # reconstruct path
    path = []
    cur = goal
    while cur is not None:
        path.append(cur)
        cur = prev[cur]
    path.reverse()
    return dist[goal], path

# Choose a start corridor and a goal corridor
start = (1, 1, 1)  # floor 1, near left corridor
goal  = (2, 3, 7)  # floor 2, near right side

cost, path = dijkstra(building, start, goal)
print("Cost:", cost)
print("Path length:", len(path))
print("First 10 nodes:", path[:10])
print("Last 10 nodes:", path[-10:])

Cost: 19.0
Path length: 12
First 10 nodes: [(1, 1, 1), (1, 2, 1), (1, 2, 2), (1, 2, 3), (1, 2, 4), (1, 1, 4), (1, 1, 5), (1, 1, 6), (2, 1, 6), (2, 1, 7)]
Last 10 nodes: [(1, 2, 2), (1, 2, 3), (1, 2, 4), (1, 1, 4), (1, 1, 5), (1, 1, 6), (2, 1, 6), (2, 1, 7), (2, 2, 7), (2, 3, 7)]


### Interpreting the path
- The path can include vertical jumps at `(floor,row,col)` that are connector cells (`Kind.CONNECTOR`).
- If you mark storage as occupied, Policy C forces routing around them.
- If you block a cell, it becomes non-walkable and paths reroute (or may become impossible).


## 10) Using your real config files (floors.updated.yaml + connectors.yaml)

In your uploaded configs:

- `floors.updated.yaml` defines:
  - `storage_penalty = 2.0`
  - default `vertical_cost_per_floor = 7.0`
  - connector symbols: `M -> mc1`, `N -> mc2`

- `connectors.yaml` defines the two monte-charges (mc1, mc2) with their coordinates per floor.

Slot region files for floor 1/2 and 3/4 exist but are currently **empty** (placeholders).
Once you digitize the plan and fill them with `[row,col]`, `apply_slot_regions()` will stamp `slot_code`
and `get_free_cells_in_region()` becomes useful for “nearest free slot in zone X”.


In [ ]:
# Load your YAML/JSON configs from your repo (config/)

import json
import yaml

# These paths are created earlier in the notebook (PROJECT_ROOT + *_path)
floors_cfg = yaml.safe_load(floors_cfg_path.read_text(encoding="utf-8"))
connectors_cfg = yaml.safe_load(connectors_cfg_path.read_text(encoding="utf-8"))
regions12 = json.loads(regions12_path.read_text(encoding="utf-8"))
regions34 = json.loads(regions34_path.read_text(encoding="utf-8"))

print("storage_penalty =", floors_cfg.get("storage_penalty"))
print("vertical_cost_per_floor =", floors_cfg.get("vertical_cost_per_floor"))
print("connector_symbols =", floors_cfg.get("connector_symbols"))

conns = connectors_cfg.get("connectors", [])
print("connectors IDs =", [c.get("id") for c in conns])

def _non_empty_region_count(regions: dict) -> int:
    return sum(1 for _, v in regions.items() if isinstance(v, list) and len(v) > 0)

print("slot regions 1/2: keys =", len(regions12), "non-empty =", _non_empty_region_count(regions12))
print("slot regions 3/4: keys =", len(regions34), "non-empty =", _non_empty_region_count(regions34))


## 11) Next steps (how this plugs into WMS services)

- **Picking Optimization**:
  - Convert pick list -> sequence of targets `(floor,r,c)` or `(slot_code)`
  - Use shortest path between them (Dijkstra/A*)
  - Minimize travel time and prioritize high-frequency SKUs

- **Storage Optimization**:
  - Candidate free slots = `get_free_cells_in_region(...)` per zone
  - Score candidates by distance from inbound / expedition zones + weight/demand
  - Output `floor + (row,col) + slot_code`

- **DB integration**:
  - `occupied` can be set from stock balances
  - `blocked` can be set from maintenance/work orders

Your current code is a clean foundation: the grid layout is stable,
while dynamic state is separated and can come from real systems.


## 12) Smoke tests (run after section 10)

This cell builds a **test building** from your real config files and runs quick checks:
- connectors create vertical edges
- Dijkstra finds a path using `building.neighbors()`
- optional: regions create `STORAGE` tiles (if your slot-region JSONs contain coordinates)

If you already have real ASCII plans, you can replace the generated plans with your own strings.


In [ ]:
from pathlib import Path
import json
import yaml
import heapq

from warehouse.loader import build_building
from warehouse.export import print_building_summary, render_floor_ascii
from warehouse.grid import walkable
from warehouse.occupancy import set_occupied

# --- 1) Reload configs (safe even if you've already done it) ---
floors_cfg = yaml.safe_load(floors_cfg_path.read_text(encoding="utf-8"))
connectors_cfg = yaml.safe_load(connectors_cfg_path.read_text(encoding="utf-8"))
regions12 = json.loads(regions12_path.read_text(encoding="utf-8"))
regions34 = json.loads(regions34_path.read_text(encoding="utf-8"))

# --- 2) Generate a test ASCII plan big enough for your connectors + slot-regions ---
def _max_rc_from_regions(regions: dict) -> tuple[int, int]:
    mr, mc = 0, 0
    for _, positions in regions.items():
        if not isinstance(positions, list):
            continue
        for pos in positions:
            if isinstance(pos, list) and len(pos) >= 2:
                mr = max(mr, int(pos[0]))
                mc = max(mc, int(pos[1]))
    return mr, mc

def _max_rc_from_connectors(connectors: list[dict]) -> tuple[int, int]:
    mr, mc = 0, 0
    for conn in connectors:
        positions = (conn or {}).get("positions", {})
        if isinstance(positions, dict):
            for _, rc in positions.items():
                if isinstance(rc, list) and len(rc) >= 2:
                    mr = max(mr, int(rc[0]))
                    mc = max(mc, int(rc[1]))
    return mr, mc

def _make_plan(rows: int, cols: int, storage_positions: list[tuple[int,int]] | None = None) -> str:
    # Fill with corridors; add a border of obstacles to visualize an envelope.
    grid = [["." for _ in range(cols)] for _ in range(rows)]
    for c in range(cols):
        grid[0][c] = "#"
        grid[rows-1][c] = "#"
    for r in range(rows):
        grid[r][0] = "#"
        grid[r][cols-1] = "#"

    if storage_positions:
        for r, c in storage_positions:
            if 0 <= r < rows and 0 <= c < cols:
                grid[r][c] = "S"

    return "\n".join("".join(row) for row in grid)

# gather storage coords from region jsons so we create STORAGE cells in the test plan
def _collect_storage_positions(regions: dict) -> list[tuple[int,int]]:
    out = []
    for _, positions in regions.items():
        if not isinstance(positions, list):
            continue
        for pos in positions:
            if isinstance(pos, list) and len(pos) >= 2:
                out.append((int(pos[0]), int(pos[1])))
    return out

connectors_list = connectors_cfg.get("connectors", [])
mr1, mc1 = _max_rc_from_connectors(connectors_list)
mr2, mc2 = _max_rc_from_regions(regions12)
mr3, mc3 = _max_rc_from_regions(regions34)

max_r = max(mr1, mr2, mr3)
max_c = max(mc1, mc2, mc3)

# Add padding so borders don't collide with important cells
rows = max(10, max_r + 3)
cols = max(10, max_c + 3)

plan12 = _make_plan(rows, cols, storage_positions=_collect_storage_positions(regions12))
plan34 = _make_plan(rows, cols, storage_positions=_collect_storage_positions(regions34))

# --- 3) Build the building ---
config = {
    "plan12": plan12,
    "plan34": plan34,
    "storage_penalty": floors_cfg.get("storage_penalty", 2.0),
    "vertical_cost_per_floor": floors_cfg.get("vertical_cost_per_floor", 5.0),
    "connector_symbols": floors_cfg.get("connector_symbols", None),
    "connectors": connectors_list,
    "slot_regions_12": regions12,
    "slot_regions_34": regions34,
}
building = build_building(config)

# --- 4) Summary + floor preview ---
print_building_summary(building)
print("\n--- Floor 1 (preview) ---")
print(render_floor_ascii(building.floors[1], show_occupied=True))

# --- 5) Vertical-edge test: neighbors from each connector position should include other floors ---
print("\n--- Vertical neighbor check from each connector (floor 1 position) ---")
for conn in connectors_list:
    cid = conn.get("id")
    pos1 = (conn.get("positions") or {}).get("1")
    if not (isinstance(pos1, list) and len(pos1) >= 2):
        continue
    f, r, c = 1, int(pos1[0]), int(pos1[1])
    verts = [(nf, nr, nc, cost) for (nf, nr, nc, cost) in building.neighbors(f, r, c) if nf != f]
    print(f"{cid} @ (floor {f}, r={r}, c={c}) vertical edges -> {verts}")

# --- 6) Dijkstra path test using building.neighbors() ---
def dijkstra(building, start, goal):
    INF = float("inf")
    dist = {start: 0.0}
    prev = {}
    pq = [(0.0, start)]
    while pq:
        d, u = heapq.heappop(pq)
        if d != dist.get(u, INF):
            continue
        if u == goal:
            break
        f, r, c = u
        for nf, nr, nc, w in building.neighbors(f, r, c):
            v = (nf, nr, nc)
            nd = d + float(w)
            if nd < dist.get(v, INF):
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))

    if goal not in dist:
        return INF, []
    path = [goal]
    cur = goal
    while cur != start:
        cur = prev[cur]
        path.append(cur)
    path.reverse()
    return dist[goal], path

# pick the first connector and route floor1 -> floor4 (same connector)
if connectors_list:
    conn0 = connectors_list[0]
    pos1 = (conn0.get("positions") or {}).get("1")
    pos4 = (conn0.get("positions") or {}).get("4")
    if isinstance(pos1, list) and isinstance(pos4, list) and len(pos1) >= 2 and len(pos4) >= 2:
        s = (1, int(pos1[0]), int(pos1[1]))
        g = (4, int(pos4[0]), int(pos4[1]))
        cost, path = dijkstra(building, s, g)
        print("\n--- Dijkstra test (first connector floor1 -> floor4) ---")
        print("connector =", conn0.get("id"))
        print("start =", s, "goal =", g)
        print("cost  =", cost)
        print("steps =", len(path))
        if path:
            print("path preview:", path[:5], "...", path[-5:])

# --- 7) Occupancy policy test (only if we have at least one STORAGE cell) ---
def find_first_storage_cell(building, floor=1):
    grid = building.floors[floor]
    for rr in range(grid.rows):
        for cc in range(grid.cols):
            if grid[rr, cc].kind.value == "STORAGE":
                return (floor, rr, cc)
    return None

p = find_first_storage_cell(building, floor=1)
if p:
    fl, rr, cc = p
    cell = building.floors[fl][rr, cc]
    print("\n--- Occupancy test on first STORAGE cell ---")
    print("Before: occupied =", cell.occupied, "| walkable =", walkable(cell))
    set_occupied(building, fl, rr, cc, True)
    print("After : occupied =", cell.occupied, "| walkable =", walkable(cell))
else:
    print("\n(No STORAGE cells found in floor 1 for this plan — occupancy test skipped.)")
